# Imports

In [ ]:
!pip install tensorboardX
# This extra module will speed up the potential fallback in the dataset download
!pip install huggingface_hub hf-transfer -q

In [ ]:
# Downlaod files, handling files and setup
import gdown
import tarfile
import urllib.request
import zipfile
import os
import gc
import sys
import subprocess
import shutil

try:
    from google.colab import drive
except ImportError:
    print("Not running in Google Colab. Drive mounting is not available.")

from pathlib import Path
from typing import List, Dict


# Data Visualization
import random
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image
import pandas as pd
import re
import cv2
from scipy.fftpack import dct
import glob

# ML pipeline
import numpy as np
import torch
from sklearn.metrics import average_precision_score, accuracy_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(device)}")

In [ ]:
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1" # Enables the download speed up

# Globals

In [ ]:
# Datasets paths
INPUT_DECOMPRESSED_IMAGES_BASE_PATH = os.path.join("input", "decompressed-dataset") # where input datasets are stored
INPUT_DATASET_BASE_PATH = os.path.join("input", "cnn-detection-dataset")
JPEG_AI_DATASETS_PATH = os.path.join("input", "JPEG_AI_datasets") # where JPEG-AI datasets for detectors finetuning are stored. They are organized in bpp-distinguished train/val/test splits.

# CNN Detector paths
CNN_DETECTOR_PATH = os.path.join("detectors", "cnn-detector")
TOOLKIT_PATH = Path.cwd() / CNN_DETECTOR_PATH / "CNNDETECTOR_SUITE"

# Universal Fake Detector (UFD) paths
UNIVFD_DETECTOR_PATH = os.path.join("detectors", "univfd")
UNIVFD_SUITE_PATH = Path.cwd() / UNIVFD_DETECTOR_PATH / "UNIVFD_SUITE"
UNIVFD_WEIGHTS = UNIVFD_SUITE_PATH / "pretrained_weights" / "fc_weights.pth"

DECOMPRESSED_IMG_BASE_PATH = os.path.join("input", "decompressed-dataset", "compressed_images")
FULL_DS_PATH = os.path.join("input")
BPP_LEVELS = ("bpp12", "bpp50", "bpp100")

# reproducibily: results-only
REUSE_PRECOMPUTED_CSV = True  # to download all the csv
REUSE_FT_CNN_DETECTOR = True  # to reuse cnn fine-tuned weights
REUSE_FT_UNIVFD = True  # to resuse univfd fine-tuned weights
REUSE_FT = REUSE_FT_CNN_DETECTOR or REUSE_FT_UNIVFD # for semplicity in handling external data, download all weights if reusing just for one detector and re-do the computation for the other

FINETUNED_MODELS_PATH = os.path.join("/content/input", "finetuned_models")

# Results save directory
RESULTS_SAVE_DIR = ""

try:
    drive.mount('/content/drive', force_remount=False)
    RESULTS_SAVE_DIR = Path("/content/drive/MyDrive/CNNDETECTOR_SUITE/ALL_Detection_Results")
except Exception as e: # This exception is raised when the notebook is not running in Google Colab, so we can safely ignore it and use a local path instead.
    print(f"Error while mounting from Google Drive: {e}\n(Is the notebook running in Google Colab?)")
    RESULTS_SAVE_DIR = Path("ALL_Detection_Results")

RESULTS_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Utils

In [ ]:
def set_seeds(seed=123):
  """Sets the random seeds for reproducibility."""
  np.random.seed(seed)
  random.seed(seed)
  torch.manual_seed(seed)

set_seeds()

# Data

To run the analysis, pristine and compressed images are downloaded.

Compression happens offline, code is provided in the external script "compress_CNN_dataset.py" because due to GPU incompatibility it would take days running on the CPU. Indeed for these computation time reason the size of the dataset taken into account has been reduced to the 10% of the original size.

Then detectors are run on compressed images, at different BPP level (0.12, 0.5, 1.0) and results, average precision and accuracy, are collected and stored in a permanent memory (csv files).

Results on pristine are later re-computed to be suitable to the portion of the dataset used in this project.

## Resource Download
All download are based on Google Drive in order to avoid time delay due to library or API limits.

Also, dependencies to run the detector all locally installed.

In [ ]:
def download_and_extract(drive_id, archive_name, output_path, post_install_cmd=None, fallback_func=None):
    """
    General utility function: downloads from GDrive, determines the extension, extracts and optionally installs dependencies.
    """
    if os.path.isdir(output_path):
        print(f"Data already in path: {output_path}; skip download.")
        # For robustness, if the path altready exists try to install the requirements as well
        if post_install_cmd:
            run_post_install(post_install_cmd)
        if output_path != "/content/input": # handle special case (due to folder structure)
            return

    print(f"Downloading {archive_name} from Google Drive...")
    os.makedirs(os.path.dirname(output_path) or "input", exist_ok=True)
    gdrive_url = f'https://drive.google.com/uc?id={drive_id}&export=download&confirm=t'

    try:
        gdown.download(gdrive_url, archive_name, quiet=False)
        print("Extracting files...")

        # Get the extension
        if archive_name.endswith('.zip'):
            with zipfile.ZipFile(archive_name, "r") as zip_ref:
                zip_ref.extractall(output_path)
        elif archive_name.endswith('.tar.gz'):
            with tarfile.open(archive_name, "r:gz") as tar_ref:
                tar_ref.extractall(output_path)
        else:
            print(f"Unsupported format: {archive_name}")
            return

        os.remove(archive_name)
        print(f"Archive {archive_name} extracted successfully.")

    except Exception as e:
        print(f"Error while downloading from Drive: {e}")
        if fallback_func:
            print("Trying the fallback...")
            fallback_func(archive_name, output_path)
        else:
            print("No fallback available.")
            return

    if post_install_cmd:
        run_post_install(post_install_cmd)

def run_post_install(cmd):
    """Install requirements via subprocess"""
    result = subprocess.run(cmd, capture_output=True)
    if result.returncode == 0:
        print("Requirements installed (or already installed)")
    else:
        print("Error while installing requirements.")
        print(result.stderr.decode('utf-8', errors='ignore'))

In [ ]:
def download_CNNDetector_dataset():
    def hf_fallback(archive_name, output_path):
        url = "https://huggingface.co/datasets/Richard-03/cnn-detection-10percent/resolve/main/cnn_dataset_reduced_10.zip"
        print("Fallback: Downloading dataset from huggingface...")
        urllib.request.urlretrieve(url, archive_name)
        with zipfile.ZipFile(archive_name, "r") as zip_ref:
            zip_ref.extractall(output_path)
        os.remove(archive_name)
        print("Dataset successfully extracted from HF.")

    download_and_extract(
        drive_id="1AnNm1MjpPQVb0-1hrj_GlTfjjDFf6tgh",
        archive_name="cnn_dataset_reduced_10.zip",
        output_path=INPUT_DATASET_BASE_PATH,
        fallback_func=hf_fallback
    )

def download_decompressed_images():
    download_and_extract(
        drive_id='1cKj7H9FX-3f_OcSCSJvpM0eUs3FIGjlA',
        archive_name="dataset_proj_cv.tar.gz",
        output_path=INPUT_DECOMPRESSED_IMAGES_BASE_PATH
    )

def download_CNNDetector_detection_suite(install_req=True):
    cmd = ["pip", "install", "-r", f"{CNN_DETECTOR_PATH}/CNNDETECTOR_SUITE/requirements.txt"] if install_req else None
    download_and_extract(
        drive_id='1tQr4tX7B-lPqEhTwAfBPF8DLObM1enit',
        archive_name="CNNDETECTOR_SUITE-zippato.zip",
        output_path=CNN_DETECTOR_PATH,
        post_install_cmd=cmd
    )

def download_UnivFD_suite(install_req=True):
    cmd = ["pip", "install", "scikit-learn", "tqdm", "ftfy", "regex"] if install_req else None
    download_and_extract(
        drive_id='14Lx_rMqf1WrAY-f8IGe0oZ_Wfxa44MQy',
        archive_name="UNIVFD_SUITE.zip",
        output_path=UNIVFD_DETECTOR_PATH,
        post_install_cmd=cmd
    )

def download_mitigation_boost_data():
    download_and_extract(
        drive_id='1QmdG-ym_aDajmhLyNY0xqgkYsfy1niyC',
        archive_name="mitigation_boost.zip",
        output_path=os.path.join("input", "mitigation")
    )

def download_JPEG_AI_datasets():
    download_and_extract(
        drive_id='1UkGzlR-9jf15Rjs2pJ6fLb0ZAadPYXED',
        archive_name="JPEG_AI_datasets.tar.gz",
        output_path="/content/input" # due to archive structure it already contains a root folder
    )

def download_finetuned_models():
    download_and_extract(
        drive_id='1I-VF_0Oe99GDv-OLVfAIh-YuY2bu8gfm',
        archive_name='finetuned_models.tar.gz',
        output_path=FINETUNED_MODELS_PATH
    )

def download_results():
    download_and_extract(
        drive_id='1-adsSA-e0ejaxRSpIkAEAj_HSBIbeKYY',
        archive_name="ALL_Detection_Results.zip",
        output_path=RESULTS_SAVE_DIR.parent
    )

In [ ]:
download_mitigation_boost_data()

In [ ]:
download_CNNDetector_dataset()

In [ ]:
download_decompressed_images()

In [ ]:
download_CNNDetector_detection_suite(install_req=True)

In [ ]:
download_UnivFD_suite()

# ===== END OF SETUP FOR UNIVFD: possible only after having installed the modules required =====

# Localize the path of UniversalFakeDetect
UFD_PATH = Path(UNIVFD_SUITE_PATH)
sys.path.insert(0, str(UFD_PATH))

# Import the necessary modules from the Universal Fake Detector. It has been placed outside the "Imports" cell since it requires the UFD_PATH to be set up first.
from models import get_model
from validate import RealFakeDataset

UFD_CKPT = UFD_PATH / "pretrained_weights" / "fc_weights.pth"

In [ ]:
download_JPEG_AI_datasets()

In [ ]:
if REUSE_PRECOMPUTED_CSV:
  download_results()

In [ ]:
if REUSE_FT:
  download_finetuned_models()

## Creation of the reduced dataset code
The following code will not be executed but is left to understand the procedure run to keep correspondance between pristine and compressed images and reduce the dataset at those only pristine images in order to speed up the downlaod process and deal with the cleanest data possible.

In [ ]:
COMPRESSED_BPP12_PATH = DECOMPRESSED_IMG_BASE_PATH + "/bpp12"
REDUCED_DATASET_BASE_PATH = Path("/content/drive/MyDrive/ReducedDatasetCVProject")

def find_underscore(string):
    return [i for i, char in enumerate(string) if char == "_"]

def create_reduced_dataset():
    pristine_base = Path(INPUT_DATASET_BASE_PATH)
    compressed_base = Path(COMPRESSED_BPP12_PATH)
    reduced_base = Path(REDUCED_DATASET_BASE_PATH)

    # find imgs in the bpp12 folders: names are all equals across folders
    ref_images = [f for f in compressed_base.rglob('*.png') if f.is_file()]

    total_imgs = len(ref_images)
    print(f"{total_imgs} compressed images found. Starting the copy...\n")

    fixed_count = 0
    copied_count = 0
    missing_count = 0

    # for each img take the relative path and copy somewhere else to keep the structure untouched
    for i, ref_path in enumerate(ref_images):

        rel_path = ref_path.relative_to(compressed_base) # folder structure without the base: es. progan/train/1_fake/img.png

        pristine_path = pristine_base / rel_path
        dest_path = reduced_base / rel_path

        if pristine_path.exists():
            # create the folder tree at the destination if it doesn't exist
            dest_path.parent.mkdir(parents=True, exist_ok=True)

            # copy the original file keepning the metadata
            shutil.copy2(pristine_path, dest_path)
            copied_count += 1
        else:
            comp_name = ref_path.name

            # fix problematic folders
            if "cyclegan/winter" in str(pristine_path) or "cyclegan/summer" in str(pristine_path):
                idxs = find_underscore(comp_name)
                orig_name = comp_name[:idxs[0]] + " " + comp_name[idxs[0]+1:]   # extra fix: in the original dataset there were ":".
                                                                                # In the transfer Windows doesn't support ":" in the filename, and has replaced them with _".
                fallback_pristine = pristine_base / rel_path.parent / orig_name
                if fallback_pristine.exists():
                    dest_path.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(fallback_pristine, dest_path)
                    copied_count += 1
                    fixed_count += 1
                    print(f"Fixed and copied: '{orig_name}' -> saved as -> '{comp_name}'")
                    continue

            # If the fallback also fails (or it wasn't a cyclegan issue)
            print(f"Image not found in the original dataset: {pristine_path}")
            missing_count += 1

        if (i + 1) % 500 == 0 or (i + 1) == total_imgs:
            print(f"Processed: {i + 1}/{total_imgs} ({(i + 1)/total_imgs*100:.1f}%)")

    print("="*50)
    print(f"Number of imgs copied: {copied_count}")
    print(f"Not found:{missing_count}")

SKIP_CELL = True # DO NOT SWITCH
if not SKIP_CELL:
  create_reduced_dataset()

In [ ]:
if not SKIP_CELL:
  # THIS bash SCRITP HAS BEEN USED TO CHECK ALL IMAGES OF THE DATASET WERE ALSO COMPRESSED AND VICEVERSA (OPTIMIZING DATA SIZE)
  BASE_DIR=DECOMPRESSED_IMG_BASE_PATH
  OUT_DIR="/content"
  REDUCED_DIR = "/content/drive/MyDrive/ReducedDatasetCVProject"

  # 'sed' to remove the whole abs path and leave only "progan/train/..."
  !find $BASE_DIR/bpp12 -type f | sed "s|^$BASE_DIR/bpp12/||" | sort > $OUT_DIR/lista_bpp12.txt
  !find $BASE_DIR/bpp50 -type f | sed "s|^$BASE_DIR/bpp50/||" | sort > $OUT_DIR/lista_bpp50.txt
  !find $BASE_DIR/bpp100 -type f | sed "s|^$BASE_DIR/bpp100/||" | sort > $OUT_DIR/lista_bpp100.txt
  !find $REDUCED_DIR -type f | sed "s|^$REDUCED_DIR/||" | sort > $OUT_DIR/lista_ridotta.txt

  # checking for differences
  !echo "--- Comparison bpp12 vs bpp50 ---"
  !diff $OUT_DIR/lista_bpp12.txt $OUT_DIR/lista_bpp50.txt

  !echo "--- Comparison bpp12 vs bpp100 ---"
  !diff $OUT_DIR/lista_bpp12.txt $OUT_DIR/lista_bpp100.txt

  !echo "--- Comparison Pristine (Reduced) vs BPP12 ---"
  !diff $OUT_DIR/lista_ridotta.txt $OUT_DIR/lista_bpp12.txt

  !echo "Check completed"

In [ ]:
if not SKIP_CELL:
  # RUN ONCE
  !zip -r -q /content/drive/MyDrive/cnn_dataset_reduced_10.zip /content/drive/MyDrive/ReducedDatasetCVProject/

## Visual inspection


In [ ]:
def show_visual_comparison(samples:int=8):
  if samples < 2:
      print("At least 2 rows!")
      samples = 2

  # Paths
  pristine_base = Path(INPUT_DATASET_BASE_PATH)
  compressed_base = Path(DECOMPRESSED_IMG_BASE_PATH)

  # Get all images from bpp12 to use as a reference for finding matches
  ref_images = list((compressed_base / 'bpp12').rglob('*.png')) # extension-proof
  selected_refs = random.choices(ref_images, k=samples)

  # Grid : samples x 4
  fig, axes = plt.subplots(samples, 4, figsize=(16, 4 * samples))
  columns = ['Pristine', 'BPP 12', 'BPP 50', 'BPP 100']

  for i, ref_path in enumerate(selected_refs):
      # Relative path from the bpp12 folder to get the sub-structure
      rel_path = ref_path.relative_to(compressed_base / 'bpp12')
      # Pristine/Main dataset and compressed one have the same structure
      pristine_path = pristine_base / rel_path
      # Row path are paths of the same img in diffent fashion: [pristine] + [1 for each BPP level]
      row_paths = [pristine_path] + [compressed_base / b / rel_path for b in BPP_LEVELS]

      """
      Path meaning:
      (fixed) compressed_base : input/decompressed-dataset/compressed_images
      (fixed) pristine_base : input/cnn-detection-dataset
      ref_path : input/decompressed-dataset/compressed_images/bpp12/progan/train/1_fake/03294.png
      rel_path : progan/train/1_fake/03294.png
      pristine_path : input/cnn-detection-dataset/progan/train/1_fake/03294.png
      """

      for j, img_path in enumerate(row_paths):
          ax = axes[i, j]
          if img_path.exists():
              img = Image.open(img_path)
              ax.imshow(img)
              if i == 0:
                  ax.set_title(columns[j], fontsize=14, fontweight='bold')
              # Label with relative path to confirm matching
              if j == 0:
                  ax.set_ylabel(f"Sample {i+1}", fontsize=12)
          else:
              print("Issue with path:", img_path)
              ax.text(0.5, 0.5, "Not Found", ha='center')
          ax.axis('off')

  plt.tight_layout()
  plt.show()

show_visual_comparison(samples = 4)

## Evaluation of Detectors
- CNNDetector model variant 0.5 (rate of JPEG compression)
- CNNDetector model variant 0.1
- Universal fake detector

### CNNDetector

In [ ]:
def get_id_from_path_and_detector(model_pth:str, detector:str):
    return "ID" + model_pth + detector

In [ ]:
# Load current results: checks on this information are used for persistency
try:
    current_global_df = pd.read_csv(Path(RESULTS_SAVE_DIR) / 'detSplit_modSplit_bppSplit_metricsSplit.csv')
    CURRENT_IDS = current_global_df["ID_Path"].tolist()
except Exception as e: # empty file or non-existing file == first execution
    print(e)
    CURRENT_IDS = []

try:
    current_pristine_df = pd.read_csv(Path(RESULTS_SAVE_DIR) / "pristine_results.csv")
    CURRENT_IDS = CURRENT_IDS + current_pristine_df["ID_Path"].tolist()
except Exception as e: # empty file or non-existing file == first execution
    print(e)
    # no change to CURRENT_IDS

print(f"Loaded {len(CURRENT_IDS)} rows")

In [ ]:
detector_pth_to_test = ["weights/blur_jpg_prob0.5.pth", "weights/blur_jpg_prob0.1.pth"] # variants of CNNDetector
cnn_compression_results = {detector: {} for detector in detector_pth_to_test} # dictionary of results for CNNDetector in both variants

def get_dataset_paths(ds_path):
    """Search for dataset paths."""
    if not os.path.exists(ds_path):
        print("Error: dataset not found:", ds_path)
        return []

    # Flat / merged case: 0_real and 1_fake are DIRECTLY inside ds_path
    if os.path.exists(os.path.join(ds_path, "0_real")) and os.path.exists(os.path.join(ds_path, "1_fake")):
        return [ds_path]

    # Nested case: 0_real / 1_fake are one level deeper (e.g. biggan/0_real)
    DATASET_TO_CHECK = []
    for root, dirs, files in os.walk(ds_path):
        dirs.sort()
        files.sort()
        for dir in dirs:
            if os.path.exists(os.path.join(root, dir, "0_real")) and os.path.exists(os.path.join(root, dir, "1_fake")):
                DATASET_TO_CHECK.append(os.path.join(root, dir))
    return sorted(DATASET_TO_CHECK)

def test_detector_cnn(model_pth, ds_path, verbose=0, is_pristine=False):
  """
  Runs CNN detector using the model and the specified path.
  Before running make sure global variable 'cnn_compression_results' is properly initialized.
  """
  # Search for dataset to run the detector on
  DATASET_TO_CHECK = get_dataset_paths(ds_path)

  for idx, d in enumerate(DATASET_TO_CHECK):
    # NO EXTRA WORK
    if get_id_from_path_and_detector(model_pth, d) in CURRENT_IDS:
        continue

    if not is_pristine:
      s = d.split("/")
      bpp = None
      bpp_idx = None

      for i, part in enumerate(s):
          m = re.match(r"bpp(\d+)(?:_.*)?$", part)
          if m:
              bpp = int(m.group(1))
              bpp_idx = i
              break

          if part == "merged_dataset":
              bpp = "merged"
              bpp_idx = i
              break

      if bpp_idx is None:
          print(f"Warning: cannot extract dataset type from path {d}, skipping.")
          continue

      next_part = s[bpp_idx + 1] if bpp_idx + 1 < len(s) else ""

      if next_part in ("train", "test", "val"):
          model = s[bpp_idx + 2] if bpp_idx + 2 < len(s) else "merged"
      else:
          model = next_part or "merged"
    else:
            # pristine dataset-specific operations
            bpp = -1
            s = d.split("/")
            model = s[-1] if len(s) == 3 else s[-2]

    # Building the data structure
    if bpp not in cnn_compression_results[model_pth]:
      cnn_compression_results[model_pth][bpp] = {}
    if model not in cnn_compression_results[model_pth][bpp]:
      cnn_compression_results[model_pth][bpp][model] = {
        "Acc": [],
        "AP": [],
        "Acc_real": [],
        "Acc_fake": [],
        "ID_Path": [] # 'detector + path' per la persistenza
      }

    out = subprocess.run(f"python {TOOLKIT_PATH}/demo_dir.py -d {d} -m {os.path.join(TOOLKIT_PATH, model_pth)} -c 256 -j 2", capture_output=True, shell=True)

    if out.returncode != 0:
      error_msg = out.stderr.decode('utf-8')
      print(f"Subprocess error at: {d}\n{error_msg}")
      continue

    str_out=str(out.stdout)
    index = str_out.find("AP")
    str_out=str_out[index:-3]
    res = str_out.split(",")
    res[0]=" "+res[0]

    print(f"Processing dataset: {d} | bpp: {bpp} | model: {model_pth}")

    for r in res:
      match = re.search(r"\d+\.\d+", r)
      if match:
        numerical_value = float(match.group())
        # Collect data
        if "Acc" in r and "real" not in r and "fake" not in r:
          cnn_compression_results[model_pth][bpp][model]["Acc"].append(numerical_value)
        elif "AP" in r:
          cnn_compression_results[model_pth][bpp][model]["AP"].append(numerical_value)
        elif "Acc (real)" in r:
          cnn_compression_results[model_pth][bpp][model]["Acc_real"].append(numerical_value)
        elif "Acc (fake)" in r:
            cnn_compression_results[model_pth][bpp][model]["Acc_fake"].append(numerical_value)
      if verbose:
        print(r)

    new_id = get_id_from_path_and_detector(model_pth, d)
    cnn_compression_results[model_pth][bpp][model]["ID_Path"].append(new_id)
    CURRENT_IDS.append(new_id)

    if verbose:
      print("-"*50)

**File naming convention**: "det{Split|Agg}_mod{Split|Agg}_bpp{Split|Agg}_metrics{Split|Agg}"

- {Split} indicates that there was no aggregation on the corresponding field.
- {Agg} indicates that there was aggregation on that field.

Examples for CNNDetector:
- 'detSplit_modSplit_bppSplit_metricsSplit': This is the most general one possible. It represents the data exactly as output by the detector; the only discarded information is whether there are model sub-folders. For example, progan/airplane or progan/bicycle are both saved as 2 distinct rows having model=progan.

- 'detSplit_modAgg_bppSplit_metricsSplit': This aggregates the models. Therefore, we will no longer have 2 rows for progan/airplane and progan/bicycle, but 1 row with the average of the metrics.

In [ ]:
def get_flat_rows(compression_results: dict) -> pd.DataFrame:
    flat_rows = []
    for detector, bpps in compression_results.items():
        for bpp, models in bpps.items():
            for model, metrics in models.items():
                acc_list = metrics["Acc"]
                ap_list = metrics["AP"]
                acc_real_list = metrics["Acc_real"]
                ap_real_list = metrics["Acc_fake"]
                identifier_list = metrics["ID_Path"]

                # The 4 lists are updated parallely, thus they have the same length
                for i in range(len(acc_list)):
                    new_row = {
                        "detector": detector,
                        "bpp": bpp,
                        "model": model,
                        "acc": acc_list[i],
                        "ap": ap_list[i],
                        "acc_real": acc_real_list[i],
                        "acc_fake": ap_real_list[i],
                        "ID_Path": identifier_list[i]
                    }
                    flat_rows.append(new_row)
    return pd.DataFrame(flat_rows)

In [ ]:
def update_global_results(new_rows: pd.DataFrame, is_pristine=False) -> pd.DataFrame:
    """Updates result files. Returns main dataframe."""
    if not is_pristine:
        filename = Path(RESULTS_SAVE_DIR) / 'detSplit_modSplit_bppSplit_metricsSplit.csv'
    else:
        filename = Path(RESULTS_SAVE_DIR) / 'pristine_results.csv'
    try:
        current_csv_state = pd.read_csv(filename)
        display(current_csv_state)
        updated_df = pd.concat([current_csv_state, new_rows], ignore_index=True)
    except: # no rows
        updated_df = pd.DataFrame(new_rows)

    updated_df.to_csv(filename, index=False) # (re-)write the global file

    if not is_pristine:
      # Aggregate data having the same model name
      full_df_aggModel = updated_df.groupby(["detector", "bpp", "model"])[["acc", "ap", "acc_real", "acc_fake"]].mean().reset_index() # .reset_index() to recover also info about the aggregators
      full_df_aggModel.to_csv(Path(RESULTS_SAVE_DIR) / 'detSplit_modAgg_bppSplit_metricsSplit.csv', index=False)

      # Aggregate data per detector and bpp; mean over models;
      full_df_aggModel_aggBpp = updated_df.groupby(["detector", "bpp"])[["acc", "ap", "acc_real", "acc_fake"]].mean().reset_index()
      full_df_aggModel_aggBpp.to_csv(Path(RESULTS_SAVE_DIR) / 'detSplit_modAgg_bppAgg_metricsSplit.csv', index=False)
    return updated_df

In [ ]:
# Run on compressed
if not REUSE_PRECOMPUTED_CSV:
  for det in detector_pth_to_test:
    test_detector_cnn(model_pth=det, ds_path=DECOMPRESSED_IMG_BASE_PATH, verbose=0)

In [ ]:
# save results from compressed images
full_df = update_global_results(get_flat_rows(cnn_compression_results)) # df of compressed compressed images tested by CNNDetector

In [ ]:
full_df.info() # data frame information check (for debugging purposes)

#### Reproducibility and comparison (CNNDetector)
The initial idea was to take the results from the previous search. But this would have introuced a bias in the baseline evaluation, since the reference dataset is the full version of the reduced version considered in this project.

Therefore we had to produce new baseline results running the detector on the prisine images as well.  

In [ ]:
cnn_compression_results = {detector: {} for detector in detector_pth_to_test} # dictionary of results for CNNDetector in both variants
if not REUSE_PRECOMPUTED_CSV:
  for det in detector_pth_to_test: # run detector on pristine images
    test_detector_cnn(model_pth=det, ds_path=INPUT_DATASET_BASE_PATH, verbose=0, is_pristine=True)

In [ ]:
pristine_full_df = update_global_results(get_flat_rows(cnn_compression_results), is_pristine=True)

In [ ]:
pristine_full_df.head(5)

### UnivFD

In [ ]:
# Update current IDs
try:
    current_global_df = pd.read_csv(Path(RESULTS_SAVE_DIR) / 'detSplit_modSplit_bppSplit_metricsSplit.csv')
    CURRENT_IDS = current_global_df["ID_Path"].tolist()
except Exception as e: # empty file or non-existing file == first execution
    print(e)
    CURRENT_IDS = []

try:
    current_pristine_df = pd.read_csv(Path(RESULTS_SAVE_DIR) / "pristine_results.csv")
    CURRENT_IDS = CURRENT_IDS + current_pristine_df["ID_Path"].tolist()
except Exception as e: # empty file or non-existing file == first execution
    print(e)
    # no change to CURRENT_IDS

print(f"Loaded {len(CURRENT_IDS)} rows")

In [ ]:
def load_ufd_model(model_name="CLIP:ViT-L/14"):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")

    model = get_model(model_name)
    state_dict = torch.load(UFD_CKPT, map_location="cpu")
    model.fc.load_state_dict(state_dict)
    print (f"UnivFD: {model_name} --- Model loaded..")
    model.eval()
    model.to(device)

    return model, device

def count_images(folder):
    exts = {".png"}
    return sum(1 for p in Path(folder).rglob("*") if p.suffix.lower() in exts)


def run_ufd_on_dataset(model, device, dataset_dir, batch_size=64):
    real_dir = Path(dataset_dir) / "0_real"
    fake_dir = Path(dataset_dir) / "1_fake"

    n_real = count_images(real_dir)
    n_fake = count_images(fake_dir)

    if n_real == 0 or n_fake == 0:
        return None

    max_sample = None if n_real == n_fake else min(n_real, n_fake) # Image equality check

    dataset = RealFakeDataset(
        real_path=str(real_dir),
        fake_path=str(fake_dir),
        data_mode="ours",
        max_sample=max_sample,
        arch="CLIP:ViT-L/14",
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )

    y_true = []
    y_pred = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)

            logits = model(imgs)
            probs = torch.sigmoid(logits).flatten().cpu().numpy()

            y_pred.extend(probs.tolist())
            y_true.extend(labels.numpy().flatten().tolist())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Binary decision
    y_pred_bin = y_pred > 0.5

    # Global metrics
    ap = average_precision_score(y_true, y_pred) * 100
    acc = accuracy_score(y_true, y_pred_bin) * 100

    # Masks to separate real and false
    is_real = (y_true == 0)
    is_fake = (y_true == 1)

    # Accuracy per class: mean on correct predictions
    acc_real = np.mean(y_pred_bin[is_real] == 0) * 100 if np.sum(is_real) > 0 else 0.0
    acc_fake = np.mean(y_pred_bin[is_fake] == 1) * 100 if np.sum(is_fake) > 0 else 0.0

    return {
        "AP": float(ap),
        "Acc": float(acc),
        "Acc_real": float(acc_real),
        "Acc_fake": float(acc_fake)
    }



In [ ]:
model, device = load_ufd_model()

In [ ]:
# EXECUTION

# I've arbitrary chosen this model, for comparison reasons.
# Here there are many options if you inspect: https://github.com/WisconsinAIVision/UniversalFakeDetect/blob/main/models/__init__.py#L36
UnivFD_models = ["CLIP:ViT-L/14"]

univfd_compression_results = {detector: {} for detector in UnivFD_models} # Same structure of cnn_compression_results: rationale is oen can run separately and update results

def test_detector_univfd(unvifd_model_name, ds_path, is_pristine=False):
      DATASET_PATHS = get_dataset_paths(ds_path)
      print(f"Total to explore: {len(DATASET_PATHS)}")
      for idx, d in enumerate(DATASET_PATHS):
          print(f"Exploring path: {d}")
          # NO EXTRA WORK
          if get_id_from_path_and_detector(unvifd_model_name, d) in CURRENT_IDS:
              continue

          if not is_pristine:
            # Search 'bpp'
            res = re.search(r"(?:^|/)bpp(\d+)(?:_[^/]*)?(?:/|$)", d)
            if res is None:
                print(f"Warning: cannot extract BPP from path {d}, skipping.")
                continue
            bpp = int(res.group(1))

            # Search 'model' name
            s = d.split("/")
            bpp_prefix = f"bpp{bpp}"
            bpp_idx = next((i for i, part in enumerate(s)
                            if part == bpp_prefix or part.startswith(bpp_prefix + "_")), None)
            if bpp_idx is None:
                print(f"Warning: cannot find bpp folder in path {d}, skipping.")
                continue
            next_part = s[bpp_idx + 1] if bpp_idx + 1 < len(s) else ""
            if next_part in ("train", "test", "val"):
                compression_model_name = s[bpp_idx + 2] if bpp_idx + 2 < len(s) else "merged"
            else:
                compression_model_name = next_part or "merged"
          else:
            # pristine dataset-specific operations
            bpp = -1
            s = d.split("/")
            compression_model_name = s[-1] if len(s) == 3 else s[-2]

          # Building the data structure
          if bpp not in univfd_compression_results[unvifd_model_name]:
            univfd_compression_results[unvifd_model_name][bpp] = {}
          if compression_model_name not in univfd_compression_results[unvifd_model_name][bpp]:
            univfd_compression_results[unvifd_model_name][bpp][compression_model_name] = {
              "Acc": [],
              "AP": [],
              "Acc_real": [],
              "Acc_fake": [],
              "ID_Path": [] # 'detector + path' for persistency (disk save)
          }

          # Run
          univfd_results_single_ds = run_ufd_on_dataset(model, device, d) # model and device loaded in the previous cell
          """
          Output of this call is:

          return {
                  "AP": float(ap),
                  "Acc": float(acc),
                  "Acc_real": float(acc_real),
                  "Acc_fake": float(acc_fake)
              }
          """

          for metric, score in univfd_results_single_ds.items():
              univfd_compression_results[unvifd_model_name][bpp][compression_model_name][metric].append(score)

          # udpate persistency
          new_id = get_id_from_path_and_detector(unvifd_model_name, d)
          univfd_compression_results[unvifd_model_name][bpp][compression_model_name]["ID_Path"].append(new_id)
          CURRENT_IDS.append(new_id)


In [ ]:
if not REUSE_PRECOMPUTED_CSV:
  for unvifd_model_name in UnivFD_models:
    test_detector_univfd(unvifd_model_name, ds_path=DECOMPRESSED_IMG_BASE_PATH, is_pristine=False)

In [ ]:
full_df = update_global_results(get_flat_rows(univfd_compression_results))

In [ ]:
full_df.info() # data frame information check (for debugging purposes)

#### Reproducibility and Comparison (UnivFD)

In [ ]:
univfd_compression_results = {detector: {} for detector in UnivFD_models} # Same structure of cnn_compression_results: rationale is that one can run separately and update results
if not REUSE_PRECOMPUTED_CSV:
  for unvifd_model_name in UnivFD_models:
    test_detector_univfd(unvifd_model_name, ds_path=INPUT_DATASET_BASE_PATH, is_pristine=True)

In [ ]:
pristine_full_df = update_global_results(get_flat_rows(univfd_compression_results), is_pristine=True)

In [ ]:
pristine_full_df.info() # data frame information check (for debugging purposes)

### Plots

In [ ]:
# focus on aggregated data
pristine_full_df = pd.read_csv(Path(RESULTS_SAVE_DIR) / "pristine_results.csv")
full_df = pd.read_csv(Path(RESULTS_SAVE_DIR) / 'detSplit_modAgg_bppAgg_metricsSplit.csv')
degradation_curve_data = full_df.groupby(["detector", "bpp"])[["acc", "ap", "acc_real", "acc_fake"]].mean().reset_index()

In [ ]:
def plot_results():
  markers = ["o", "s", "^"]
  colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
  # from collected data
  pristine_avg_res = pristine_full_df.groupby(["detector"])[["acc", "ap", "acc_real", "acc_fake"]].mean().reset_index()

  for idx, model_name in enumerate(degradation_curve_data["detector"].unique()):

      bpp = degradation_curve_data["bpp"].unique()
      ap = degradation_curve_data[degradation_curve_data["detector"] == model_name]["ap"]
      acc = degradation_curve_data[degradation_curve_data["detector"] == model_name]["acc"]
      ax1.plot(bpp, ap, marker=markers[idx], color=colors[idx], label=model_name)
      ax2.plot(bpp, acc, marker=markers[idx], color=colors[idx], label=model_name)

      # prisine comparison
      p_ap = pristine_avg_res.query("detector == @model_name")["ap"].item()
      p_acc = pristine_avg_res.query("detector == @model_name")["acc"].item()
      ax1.axhline(y=p_ap, color=colors[idx], linestyle='--', alpha=0.6)
      ax2.axhline(y=p_acc, color=colors[idx], linestyle='--', alpha=0.6)

  # some aesthetics
  ax1.set_title("AP comparison")
  ax1.set_xlabel("bpp")
  ax1.set_ylabel("AP")
  ax2.set_title("Acc comparison")
  ax2.set_xlabel("bpp")
  ax2.set_ylabel("Acc")

  ax1.set_xlim(5, 105)
  ax2.set_xlim(5, 105)
  ax1.set_xticks([12, 50, 100])
  ax2.set_xticks([12, 50, 100])
  ax1.grid(True, linestyle=':', alpha=0.7)
  ax2.grid(True, linestyle=':', alpha=0.7)

  # readibility
  ax1.set_title("AP comparison")
  ax1.set_xlabel("bpp")
  ax1.set_ylabel("AP")
  ax2.set_title("Acc comparison")
  ax2.set_xlabel("bpp")
  ax2.set_ylabel("Acc")
  ax2.legend(loc="lower right")

  plt.tight_layout()
  plt.show()



plot_results()

# Frequency Analysis

## Average Power Spectrum

This plot shows the visual heatmap of frequency energy computed via the Fourier Transform. The low frequencies (representing homogeneous areas) are located at the center, while the high frequencies (representing sharp details and noise) are spread toward the edges. Deepfakes—especially those based on GANs and CNNs—often exhibit anomalous high-frequency peaks caused by upsampling operations, which typically appear as grid-like artifacts. Analyzing and comparing these spectra between real and fake images is crucial to explain the drop in detection performance.

The avg_power_spectrum function takes the path of a folder as input, computes the power spectrum for each image inside it, and then returns their average.

In [ ]:
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['font.family'] = 'STIXGeneral'
def power_spectrum(img):
  fft_2d = np.fft.fft2(img)
  fft_shifted = np.fft.fftshift(fft_2d)
  power_spectrum = np.abs(fft_shifted) ** 2
  power_spectrum_log = np.log10(1 + power_spectrum)
  return power_spectrum_log

def display_power_spectrum_single(ps,name=None):
  plt.figure(figsize=(8, 4))
  plt.subplot(1, 2, 2)
  plt.imshow(ps, cmap="inferno")
  if name==None:
    plt.title("Power Spectrum")
  else:
    plt.title("Power Spectrum: "+name)
  plt.colorbar(label="Log10(Power)")
  plt.axis("off")

  plt.tight_layout()
  plt.show()

def display_power_spectrum(ps_list, name_list=None):
    num_plots = len(ps_list)
    plt.figure(figsize=(4 * num_plots, 4))
    for i in range(num_plots):
        plt.subplot(1, num_plots, i + 1)
        ps = ps_list[i]
        plt.imshow(ps, cmap="inferno")
        if name_list and i < len(name_list) and name_list[i] is not None:
            plt.title("Power Spectrum: " + str(name_list[i]),fontsize=12)
        else:
            plt.title("Power Spectrum")
        plt.colorbar(label="Log10(Power)")
        plt.axis("off")
    plt.tight_layout()
    plt.show()


def avg_power_spectrum(folder,max_count=99999999):
  counter=0
  ps_list = []
  for image_path in folder.iterdir():
        if image_path.is_file():
          img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
          if img is None:
            raise FileNotFoundError(
                f"No image found at : {image_path}"
            )
          ps = power_spectrum(img)
          ps_list.append(ps)
          counter+=1
          if counter>=max_count:
            break
  if len(ps_list) == 0:
    return []
  avg_ps = np.mean(ps_list, axis=0)
  return avg_ps



Average power spectrum computed on a given folder.

In [ ]:
images_number=200

dataset_name='gaugan'

print("differences between compressed and original")
folder = f"/content/input/decompressed-dataset/compressed_images/bpp100/{dataset_name}/0_real"
name1="/".join(folder.split("/")[6:])
ps1=avg_power_spectrum(Path(folder),max_count=images_number)
folder = f"/content/input/cnn-detection-dataset/{dataset_name}/0_real"
name2="/".join(folder.split("/")[4:])
ps2=avg_power_spectrum(Path(folder),max_count=images_number)
folder = f"/content/input/decompressed-dataset/compressed_images/bpp100/{dataset_name}/1_fake"
name3="/".join(folder.split("/")[6:])
ps3=avg_power_spectrum(Path(folder),max_count=images_number)
folder = f"/content/input/cnn-detection-dataset/{dataset_name}/1_fake"
name4="/".join(folder.split("/")[4:])
ps4=avg_power_spectrum(Path(folder),max_count=images_number)
display_power_spectrum([ps1,ps2,ps3,ps4],[name1,name2,name3,name4])

print("\ndifferences between real compressed at different bpp")
folder = f"/content/input/decompressed-dataset/compressed_images/bpp12/{dataset_name}/0_real"
name1="/".join(folder.split("/")[6:])
ps1=avg_power_spectrum(Path(folder),max_count=images_number)
folder = f"/content/input/decompressed-dataset/compressed_images/bpp50/{dataset_name}/0_real"
name2="/".join(folder.split("/")[6:])
ps2=avg_power_spectrum(Path(folder),max_count=images_number)
folder = f"/content/input/decompressed-dataset/compressed_images/bpp100/{dataset_name}/0_real"
name3="/".join(folder.split("/")[6:])
ps3=avg_power_spectrum(Path(folder),max_count=images_number)
display_power_spectrum([ps1,ps2,ps3],[name1,name2,name3])

print("\ndifferences between fake compressed at different bpp")
folder = f"/content/input/decompressed-dataset/compressed_images/bpp12/{dataset_name}/1_fake"
name1="/".join(folder.split("/")[6:])
ps1=avg_power_spectrum(Path(folder),max_count=images_number)
folder = f"/content/input/decompressed-dataset/compressed_images/bpp50/{dataset_name}/1_fake"
name2="/".join(folder.split("/")[6:])
ps2=avg_power_spectrum(Path(folder),max_count=images_number)
folder = f"/content/input/decompressed-dataset/compressed_images/bpp100/{dataset_name}/1_fake"
name3="/".join(folder.split("/")[6:])
ps3=avg_power_spectrum(Path(folder),max_count=images_number)
display_power_spectrum([ps1,ps2,ps3],[name1,name2,name3])

Average power spectrum for all the compressed images

In [ ]:
ps_to_show = 4
counter=0

for root, dirs, files in os.walk(DECOMPRESSED_IMG_BASE_PATH):
  dirs.sort()
  files.sort()
  if "0_real" in root or "1_fake" in root:
    ps=avg_power_spectrum(Path(root))
    if len(ps)>0:
      name="/".join(root.split("/")[3:])
      display_power_spectrum_single(ps,name=name)
      counter+=1
      if counter>=ps_to_show:
        break



## Azimuthal Average

 This plot represents a "simplified" version of the 2D power spectrum. Instead of a 2D image, it provides a 1D line graph. It is computed by averaging the energy over concentric rings, moving from the center (frequency $0$) outward to the maximum frequency ($f_{max}$). It shows exactly how much energy the image contains as a function of the distance from the center.The avg_azimuthal_average function takes the path of a folder as input, calculates the azimuthal average graph for each image inside it, and then returns their overall average.


In [ ]:
def azimuthal_average(img):
  ps_2d=power_spectrum(img)
  h, w = ps_2d.shape
  center_y, center_x = h // 2, w // 2
  y, x = np.indices((h, w))
  r = np.sqrt((x - center_x)**2 + (y - center_y)**2)
  r_int = r.astype(int)

  unique_rays = np.arange(0, r_int.max() + 1)
  sum_for_rays = np.bincount(r_int.ravel(), weights=ps_2d.ravel())
  count_for_rays = np.bincount(r_int.ravel())
  count_for_rays[count_for_rays == 0] = 1
  profile_1d = sum_for_rays / count_for_rays

  return unique_rays, profile_1d

def avg_azimuthal_average(folder,max_images=999999999):
  if(type(folder)==str):
    folder=Path(folder)
  counter=0
  rays_list = []
  profile_list=[]
  for image_path in folder.iterdir():
        if image_path.is_file():
          img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
          if img is None:
            raise FileNotFoundError(
                f"No image found at : {image_path}"
            )
          rays, profile = azimuthal_average(img)
          rays_list.append(rays)
          profile_list.append(profile)
          counter+=1
          if counter>=max_images:
            break
  if len(rays_list) == 0:
    return []
  avg_rays = np.mean(rays_list, axis=0)
  avg_profile = np.mean(profile_list, axis=0)
  return avg_rays,avg_profile



Note the peaks on high frequencies.

In [ ]:
dataset_name='imle'
autenticity= '1_fake'
#autenticity= '0_real'
folders = [f"/content/input/decompressed-dataset/compressed_images/bpp12/{dataset_name}/{autenticity}",
           f"/content/input/decompressed-dataset/compressed_images/bpp50/{dataset_name}/{autenticity}",
           f"/content/input/decompressed-dataset/compressed_images/bpp100/{dataset_name}/{autenticity}",
           f"/content/input/cnn-detection-dataset/{dataset_name}/{autenticity}"]
max_images=999999999

colors = list(mcolors.TABLEAU_COLORS.keys())
plt.figure(figsize=(8, 5))
for i,folder in enumerate(folders):
  rays,profile=avg_azimuthal_average(folder,max_images=max_images)
  name="/".join(folder.split("/")[5:])
  if "bpp" not in name:
    name = "uncompressed "+dataset_name+"/"+name
  plt.plot(rays, np.log10(1 + profile),
         color=colors[i%len(colors)], linestyle='-', linewidth=2, label=name)





plt.title(f"Azimuthal Average", fontsize=14, fontweight='bold')
plt.xlabel("spatial frequency (dist. from the center in pixel)", fontsize=11)
plt.ylabel("Log10(avg energy)", fontsize=11)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()




## DCT Coefficients

If we divide the image into $8 \times 8$ blocks, the Discrete Cosine Transform (DCT) produces 64 coefficients per block (1 DC, 63 AC). Plotting the histogram of a specific high-frequency coefficient allows you to see how these mathematical values are distributed.

In [ ]:
def DCT_coefficients(img):
  if img is None:
    raise ValueError("No image. check the path.")
  #ensure the image can be divide in 8x8 blocks
  height, width = img.shape
  h = (height//8)*8
  w = (width//8)*8
  img=img[:h,:w]
  img_float = img.astype(np.float32) #convert the image in float32
  dct_coefficients = np.zeros_like(img_float)
  for r in range(0, h, 8):
    for c in range(0, w, 8):
      pixel_block=img_float[r:r+8,c:c+8]
      dct_block=cv2.dct(pixel_block)
      dct_coefficients[r:r+8,c:c+8]=dct_block
  return dct_coefficients

def compute_DCT_coefficient_histogram(img, u_target, v_target, bins_count=100):
    h, w = img.shape
    h_new, w_new = (h // 8) * 8, (w // 8) * 8
    img = img[:h_new, :w_new]
    img_float = img.astype(np.float32) - 128.0
    dct_blocks = np.zeros_like(img_float)
    for r in range(0, h_new, 8):
        for c in range(0, w_new, 8):
            dct_blocks[r:r+8, c:c+8] = cv2.dct(img_float[r:r+8, c:c+8])
    extracted_coefficients = []
    for r in range(0, h_new, 8):
        for c in range(0, w_new, 8):
            val = dct_blocks[r + u_target, c + v_target]
            extracted_coefficients.append(val)
    extracted_coefficients = np.array(extracted_coefficients)
    counts, bin_edges = np.histogram(extracted_coefficients, bins=bins_count, density=True)
    return counts, bin_edges

def avg_DCT_coefficients_histogram(folder,u,v,max_images=999999):
  if(type(folder)==str):
    folder=Path(folder)
    counter=0
    counts_list = []
    bin_edges_list=[]
    for image_path in folder.iterdir():
          if image_path.is_file():
            img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
              raise FileNotFoundError(
                  f"No image found at : {image_path}"
              )
            counts, bin_edges = compute_DCT_coefficient_histogram(img,u,v)
            counts_list.append(counts)
            bin_edges_list.append(bin_edges)
            counter+=1
            if counter>=max_images:
              break
    if len(counts_list) == 0:
      return []
    counts_avg = np.mean(counts_list, axis=0)
    bin_edges_avg = np.mean(bin_edges_list, axis=0)
    return counts_avg,bin_edges_avg


In [ ]:
N_IMAGES = 200

def find_leaf_dirs(base_dir):
    """
    Recursively searches all folders containing both 0_real
    and 1_fake as direct subfolders. This works for both flat (e.g., imle/0_real)
    and nested (e.g., progan/airplane/0_real) datasets.
    """
    base_dir = Path(base_dir)
    leaves = []
    if not base_dir.exists():
        return leaves
    for root, dirs, files in os.walk(base_dir):
        dirs.sort()
        if "0_real" in dirs and "1_fake" in dirs:
            leaves.append(Path(root))
    return leaves


def get_dataset_base(dataset_name, bpp):
    if bpp == "pristine":
        return Path(INPUT_DATASET_BASE_PATH) / dataset_name
    return Path(DECOMPRESSED_IMG_BASE_PATH) / bpp / dataset_name


def aggregate_power_spectrum(dataset_name, bpp, authenticity, max_count=N_IMAGES):
    leaves = find_leaf_dirs(get_dataset_base(dataset_name, bpp))
    spectra = []
    for leaf in leaves:
        ps = avg_power_spectrum(leaf / authenticity, max_count=max_count)
        if len(ps) > 0:
            spectra.append(ps)
    return np.mean(spectra, axis=0) if spectra else []


def aggregate_azimuthal_average(dataset_name, bpp, authenticity, max_images=N_IMAGES):
    leaves = find_leaf_dirs(get_dataset_base(dataset_name, bpp))
    profiles = []
    for leaf in leaves:
        result = avg_azimuthal_average(leaf / authenticity, max_images=max_images)
        if len(result) == 0:
            continue
        _, profile = result
        profiles.append(profile)
    if not profiles:
        return []
    min_len = min(len(p) for p in profiles)   # categorie diverse -> dimensioni diverse
    avg_profile = np.mean([p[:min_len] for p in profiles], axis=0)
    return np.arange(min_len), avg_profile


def aggregate_DCT_histogram(dataset_name, bpp, authenticity, u, v, max_images=N_IMAGES):
    leaves = find_leaf_dirs(get_dataset_base(dataset_name, bpp))
    counts_list, edges_list = [], []
    for leaf in leaves:
        result = avg_DCT_coefficients_histogram(str(leaf / authenticity), u, v, max_images=max_images)
        if len(result) == 0:
            continue
        counts, edges = result
        counts_list.append(counts)
        edges_list.append(edges)
    if not counts_list:
        return []
    return np.mean(counts_list, axis=0), np.mean(edges_list, axis=0)

In [ ]:
U, V = 5, 5   # high-frequency DCT coefficients (8x8 blocks)

def display_DCT_histogram_overlay(hist_dict, u, v, title=""):
    plt.figure(figsize=(9, 5))
    for label, (counts, bin_edges) in hist_dict.items():
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        plt.plot(bin_centers, counts, linewidth=2, label=label)
    plt.title(f"Coefficients distribution DCT (u={u}, v={v}) — {title}", fontsize=13)
    plt.xlabel("Coefficients value")
    plt.ylabel("Probability density")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.show()

In [ ]:
DATASETS_TO_PLOT = ["progan", "gaugan", "imle"]   # adapt to the datasets you have in your input folder

for dataset in DATASETS_TO_PLOT:
    for authenticity, label in zip(["1_fake", "0_real"], ["FAKE", "REAL"]):
        hist_dict = {}
        for bpp in ["pristine"] + list(BPP_LEVELS):
            result = aggregate_DCT_histogram(dataset, bpp, authenticity, U, V, max_images=N_IMAGES)
            if len(result) == 0: continue
            hist_dict[bpp] = result
        display_DCT_histogram_overlay(hist_dict, U, V, title=f"{dataset} - {label}")

# Network

# Compression-aware mitigation
First we search for the category where the drop in performance is larger to have clear evidence of the improvement. Definitely 'stylegan2' shows the largest drop (from $99\%$ on pristine to $\approx$ $52\%$ AP).

We implement it for the case of $BPP=12$ being the one that has more margin of improvements.

In [ ]:
# search for the category/model with the largest worsening in AP
category_df = pd.read_csv(Path(RESULTS_SAVE_DIR) / "detSplit_modAgg_bppSplit_metricsSplit.csv")
category_df_12 = category_df.query("bpp==12")
pristine_full_df = pd.read_csv(Path(RESULTS_SAVE_DIR) / "pristine_results.csv")
pristine_full_df_model_12 = pristine_full_df.groupby(by=["model"]).mean(numeric_only=True).reset_index()

merged_df = pd.merge(
    category_df_12,
    pristine_full_df_model_12[['model', 'ap']],
    on='model',
    how='inner', # inner joint to ensure computation is done only on rows with the 'model' appearing in both
    suffixes=('_distorted', '_pristine')
)

merged_df['ap_diff'] = merged_df['ap_pristine'] - merged_df['ap_distorted']
merged_df = merged_df.sort_values(by='ap_diff', ascending=False)
merged_df.head()

In [ ]:
def frequency_boost_channel(channel, gain=2.0, freq_threshold=2):
    """
    Amplifies the high-frequency AC DCT coefficients of a 2D channel.
    u+v > freq_threshold defines what we consider "high frequency"
    in the 8x8 block: (0,0)=DC, progressively higher values = higher frequencies.
    """
    h, w = channel.shape
    h8, w8 = (h // 8) * 8, (w // 8) * 8
    out = channel.copy().astype(np.float32)

    u, v = np.indices((8, 8))
    high_freq_mask = (u + v) > freq_threshold

    for r in range(0, h8, 8):
        for c in range(0, w8, 8):
            block = out[r:r+8, c:c+8] - 128.0
            dct_block = cv2.dct(block)
            dct_block[high_freq_mask] *= gain
            out[r:r+8, c:c+8] = cv2.idct(dct_block) + 128.0

    return np.clip(out, 0, 255).astype(np.uint8)


def frequency_boost_image(img_bgr, gain=2.0, freq_threshold=2):
    """Boost applied only to the Y luminance channel (YCrCb space)."""
    ycrcb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)
    y, cr, cb = cv2.split(ycrcb)
    y_boosted = frequency_boost_channel(y, gain=gain, freq_threshold=freq_threshold)
    boosted = cv2.merge([y_boosted, cr, cb])
    return cv2.cvtColor(boosted, cv2.COLOR_YCrCb2BGR)


def apply_frequency_boost_to_folder(src_folder, dst_folder, gain=2.0, freq_threshold=2):
    src_folder, dst_folder = Path(src_folder), Path(dst_folder)
    dst_folder.mkdir(parents=True, exist_ok=True)
    exts = {'.jpg', '.jpeg', '.png', '.webp'}
    for img_path in src_folder.iterdir():
        if img_path.suffix.lower() not in exts:
            continue
        img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        if img is None:
            continue
        boosted = frequency_boost_image(img, gain=gain, freq_threshold=freq_threshold)
        cv2.imwrite(str(dst_folder / img_path.name), boosted)

In [ ]:
CATEGORY = "stylegan2"          # category with largest AP decrease
FREQ_THRESHOLD = 2
GAINS_TO_TEST = [1.5, 2.0, 3.0, 10]  # ablation: what is the gain better recovers the signal?

In [ ]:
src_base = get_dataset_base(CATEGORY, "bpp12")

# extract only
for gain in GAINS_TO_TEST:
    if os.path.exists(f"input/mitigation/mitigation_boost/gain_{gain}"):
      continue
    dst_base = Path(f"input/mitigation/mitigation_boost/gain_{gain}/bpp12") / CATEGORY
    for leaf in find_leaf_dirs(src_base):
        rel = leaf.relative_to(src_base)
        for authenticity in ["0_real", "1_fake"]:
            apply_frequency_boost_to_folder(
                leaf / authenticity,
                dst_base / rel / authenticity,
                gain=gain, freq_threshold=FREQ_THRESHOLD
            )
    print(f"✅ Boost gain={gain} completed for {CATEGORY}")

In [ ]:
for gain in GAINS_TO_TEST:
    dst_base = Path(f"input/mitigation/mitigation_boost/gain_{gain}/bpp12") / CATEGORY

    for authenticity, label in zip(["1_fake", "0_real"], ["FAKE", "REAL"]):
        hist_dict = {}

        result_p = aggregate_DCT_histogram(CATEGORY, "pristine", authenticity, U, V, max_images=N_IMAGES)
        if len(result_p): hist_dict["pristine"] = result_p

        result_c = aggregate_DCT_histogram(CATEGORY, "bpp12", authenticity, U, V, max_images=N_IMAGES)
        if len(result_c): hist_dict["bpp12 (no boost)"] = result_c

        leaves = find_leaf_dirs(dst_base)
        boosted_hists = []
        for leaf in leaves:
            r = avg_DCT_coefficients_histogram(str(leaf / authenticity), U, V, max_images=N_IMAGES)
            if len(r): boosted_hists.append(r)
        if boosted_hists:
            counts_avg = np.mean([c for c, e in boosted_hists], axis=0)
            edges_avg = np.mean([e for c, e in boosted_hists], axis=0)
            hist_dict[f"bpp12 + boost (gain={gain})"] = (counts_avg, edges_avg)

        display_DCT_histogram_overlay(hist_dict, U, V, title=f"{CATEGORY} - {label} - gain={gain}")

In [ ]:
if False: # this cell has been left just for tracking purpose; it's not meant to be run
    !zip -r -q /content/drive/MyDrive/mitigation_boost.zip /content/input/mitigation/mitigation_boost

In [ ]:
MITIGATION_RESULTS = []
bpp_of_interest = 12

def test_gain(gain, bpp_of_interest):
    category_path = Path(f"input/mitigation/mitigation_boost/gain_{gain}/bpp{bpp_of_interest}") / CATEGORY

    # CNNDetector
    for model_pth in detector_pth_to_test:
        test_detector_cnn(model_pth, category_path, verbose=0, is_pristine=False)

        MITIGATION_RESULTS.append({
            'condition': f'bpp12+boost_gain{gain}',
            'gain': gain,
            'detector': model_pth,
            'model': CATEGORY,
            'ap': np.array(cnn_compression_results[model_pth][bpp_of_interest][CATEGORY]['AP']).mean(),
            'acc': np.array(cnn_compression_results[model_pth][bpp_of_interest][CATEGORY]['Acc']).mean()
        })

    # UniFD
    for unvifd_model_name in UnivFD_models:
        test_detector_univfd(unvifd_model_name, category_path, is_pristine=False)
        MITIGATION_RESULTS.append({
            'condition': f'bpp12+boost_gain{gain}',
            'gain': gain,
            'detector': unvifd_model_name,
            'model': CATEGORY,
            'ap': np.array(univfd_compression_results[unvifd_model_name][bpp_of_interest][CATEGORY]['AP']).mean(),
            'acc': np.array(univfd_compression_results[unvifd_model_name][bpp_of_interest][CATEGORY]['Acc']).mean()
        })


In [ ]:
# computation saving
try: # this path is walked if REUSE_PRECOMPUTED_CSV = True
    mitigation_df = pd.read_csv(Path(RESULTS_SAVE_DIR) / "mitigation_results.csv")
    MITIGATION_RESULTS = mitigation_df.to_dict(orient='records')
except:
    mitigation_df = None
    print("No data found. Regeneration from scratch...")

# run for different gains
for gain in GAINS_TO_TEST:
    if mitigation_df is not None:
      if gain in mitigation_df['gain'].unique():
        continue
    cnn_compression_results = {detector: {} for detector in detector_pth_to_test} # RESET for code-resuse purpose
    CURRENT_IDS = [] # RESET for code-reuse purpose
    univfd_compression_results = {detector: {} for detector in UnivFD_models} # RESET for code-reuse purpose

    test_gain(gain, bpp_of_interest=12)

In [ ]:
df_mitigation = pd.DataFrame(MITIGATION_RESULTS)
df_mitigation.to_csv(Path(RESULTS_SAVE_DIR) / "mitigation_results.csv", index=False)
df_mitigation

**Consideration**: we implemented a frequency-aware preprocessing strategy, and we obeserved that the recover of spectral statistics (showed by the DCT plot above where the "boost" curve approximates the pristine one) doesn't tranlate in a revovery in a significant improvement of the detector performance (just a few AP point). This suggests that the detectors act on featrues of a superior order with respect to the statistics of the DCT coefficients.

In [ ]:
category_agg_det = category_df[(category_df["model"] == CATEGORY) & (category_df["bpp"] == 12)].groupby(by=["detector"]).mean(numeric_only=True).reset_index()
category_agg_det.head()

In [ ]:
pristine_agg_det = pristine_full_df[pristine_full_df["model"] == CATEGORY].groupby(by=["detector"]).mean(numeric_only=True).reset_index()
pristine_agg_det.head()

In [ ]:
# Collecting data for a comparative table

detectors = [
    "weights/blur_jpg_prob0.5.pth",
    "weights/blur_jpg_prob0.1.pth",
    "CLIP:ViT-L/14",
]

rows = []

# pristine
row = pristine_agg_det.set_index("detector").loc[detectors, "ap"]
row.name = "Pristine"
rows.append(row)

# bpp12 no mitigation
row = category_agg_det.set_index("detector").loc[detectors, "ap"]
row.name = "bpp12 no mitigation"
rows.append(row)

# bpp12 + boost
for gain in [1.5, 2.0, 3.0, 10.0]:
    row = (
        df_mitigation[df_mitigation["gain"] == gain]
        .set_index("detector")
        .loc[detectors, "ap"]
    )
    row.name = f"bpp12 + boost gain={gain}"
    rows.append(row)

final_table = pd.DataFrame(rows).reset_index()
final_table = final_table.rename(columns={"index": "condition"})

final_table

## Frequency Masking

If the high frequencies contain only compression noise that confuses the detector, they can be eliminated, thereby removing the variance introduced by JPEG AI. The model will be forced to rely exclusively on mid-frequencies (such as the $u=3, v=3$ block), where analysis has shown that a statistical gap between Real and Fake persists even under compression.

In [ ]:
#apply a circular mask on the DFT of an image, resulting in a low pass filter
def apply_frequency_mask(image_input, cutoff_radius):
  if isinstance(image_input, (str, Path)):
      img = cv2.imread(str(image_input), cv2.IMREAD_GRAYSCALE)
      if img is None:
          raise ValueError(f"Unable to load image: {image_input}")
  elif isinstance(image_input, np.ndarray):
      # If the image is colored (RGB/BGR), convert it to grayscale
      if len(image_input.shape) == 3:
          img = cv2.cvtColor(image_input, cv2.COLOR_BGR2GRAY)
      else:
          img = image_input.copy()
  else:
      raise TypeError("Input must be a file path (str/Path) or a numpy array.")


  f_transform = np.fft.fft2(img)

  f_shift = np.fft.fftshift(f_transform)


  rows, cols = img.shape
  center_row, center_col = rows // 2, cols // 2

  # create a grid of coordinates to calculate the distance from the center
  Y, X = np.ogrid[:rows, :cols]
  dist_from_center = np.sqrt((X - center_col)**2 + (Y - center_row)**2)

  # boolean mask
  mask = dist_from_center <= cutoff_radius

  f_shift_masked = f_shift * mask


  f_ishift = np.fft.ifftshift(f_shift_masked)
  img_back = np.fft.ifft2(f_ishift)
  img_filtered = np.abs(img_back)

  img_filtered = cv2.normalize(img_filtered, None, 0, 255, cv2.NORM_MINMAX)
  return img_filtered.astype(np.uint8)


def visualize_frequency_filtering(original_img, filtered_img):


    f_transform_orig = np.fft.fft2(original_img)
    f_shift_orig = np.fft.fftshift(f_transform_orig)
    magnitude_spectrum_orig = 20 * np.log1p(np.abs(f_shift_orig))


    f_transform_filt = np.fft.fft2(filtered_img)
    f_shift_filt = np.fft.fftshift(f_transform_filt)
    magnitude_spectrum_filt = 20 * np.log1p(np.abs(f_shift_filt))


    plt.figure(figsize=(10, 6))

    # original Image
    plt.subplot(2, 2, 1)
    plt.imshow(original_img, cmap='gray')
    plt.title('Original Image (Spatial Domain)', fontsize=14)
    plt.axis('off')

    # filtered Image
    plt.subplot(2, 2, 2)
    plt.imshow(filtered_img, cmap='gray')
    plt.title('Filtered Image (Spatial Domain)', fontsize=14)
    plt.axis('off')

    # original Spectrum
    plt.subplot(2, 2, 3)
    plt.imshow(magnitude_spectrum_orig, cmap='magma')
    plt.title('Original Frequency Spectrum', fontsize=14)
    plt.colorbar(label='Log Intensity', fraction=0.046, pad=0.04)
    plt.axis('off')

    # filtered Spectrum
    plt.subplot(2, 2, 4)
    plt.imshow(magnitude_spectrum_filt, cmap='magma')
    plt.title('Filtered Frequency Spectrum', fontsize=14)
    plt.colorbar(label='Log Intensity', fraction=0.046, pad=0.04)
    plt.axis('off')

    plt.tight_layout()
    plt.show()

def batch_process_dataset(source_dir, target_dir, cutoff_radius, valid_exts=('.png', '.jpg', '.jpeg'), max_images=None):
    tgt_path = Path(target_dir)

    if tgt_path.exists():
        print(f"Target directory already exists: {target_dir}. Aborting.")
        return

    src_path = Path(source_dir)
    processed_count = 0

    for img_path in src_path.rglob('*'):
        if max_images is not None and processed_count >= max_images:
            break

        if img_path.is_file() and img_path.suffix.lower() in valid_exts:
            rel_path = img_path.relative_to(src_path)
            out_path = tgt_path / rel_path

            out_path.parent.mkdir(parents=True, exist_ok=True)

            filtered_img = apply_frequency_mask(img_path, cutoff_radius)
            cv2.imwrite(str(out_path), filtered_img)

            processed_count += 1

In [ ]:
img = cv2.imread("/content/input/decompressed-dataset/compressed_images/bpp12/stylegan2/cat/0_real/02045.png", cv2.IMREAD_GRAYSCALE)
filtered_img = apply_frequency_mask(img,80)
visualize_frequency_filtering(img, filtered_img)

In [ ]:
bpp=12
dataset= "stylegan2"
possible_cutoff_radius = [10,20]

for cutoff_radius in possible_cutoff_radius:
  folder = f"bpp{bpp}/{dataset}/"
  batch_process_dataset(source_dir=f"/content/input/decompressed-dataset/compressed_images/{folder}",
                        target_dir=f"/content/input/mitigation/frequency_masking/{folder}/cutOff{cutoff_radius}/",
                        cutoff_radius=cutoff_radius, valid_exts=('.png'))


### Testing detector on new dataset

In [ ]:
FREQUENCY_MASKING_RESULTS = []
bpp_of_interest = 12
dataset= "stylegan2"

def test_frequency_masking(cutoff_radius_of_interest, bpp_of_interest,dataset):
    category_path = Path(f"input/mitigation/frequency_masking/bpp{bpp_of_interest}/{dataset}/cutOff{cutoff_radius_of_interest}")
    # CNNDetector
    for model_pth in detector_pth_to_test:
        test_detector_cnn(model_pth, category_path, verbose=0, is_pristine=False)

        FREQUENCY_MASKING_RESULTS.append({
            'bpp':bpp_of_interest,
            'cut off': cutoff_radius_of_interest,
            'detector': model_pth,
            'model': dataset,
            'ap': np.array(cnn_compression_results[model_pth][bpp_of_interest][CATEGORY]['AP']).mean(),
            'acc': np.array(cnn_compression_results[model_pth][bpp_of_interest][CATEGORY]['Acc']).mean()
        })

    # UniFD
    for unvifd_model_name in UnivFD_models:
        test_detector_univfd(unvifd_model_name, category_path, is_pristine=False)
        FREQUENCY_MASKING_RESULTS.append({
            'bpp':bpp_of_interest,
            'cut off': cutoff_radius_of_interest,
            'detector': unvifd_model_name,
            'model': dataset,
            'ap': np.array(univfd_compression_results[unvifd_model_name][bpp_of_interest][CATEGORY]['AP']).mean(),
            'acc': np.array(univfd_compression_results[unvifd_model_name][bpp_of_interest][CATEGORY]['Acc']).mean()
        })

In [ ]:
if REUSE_PRECOMPUTED_CSV:
  df_mitigation_freq_masking = pd.read_csv(Path(RESULTS_SAVE_DIR) / "mitigation_results_freq_masking.csv")
else:
  for co in possible_cutoff_radius:
    test_frequency_masking(co,bpp_of_interest,dataset)

  df_mitigation_freq_masking = pd.DataFrame(FREQUENCY_MASKING_RESULTS)



  new_row_dict1 = {
      'bpp': '12',
      'cut off': 'NAN',
      'detector': 'CLIP:ViT-L/14',
      'model': 'stylegan2',
      'acc': 57.587626,
      'ap':54.967088
  }
  new_row_dict2 = {
      'bpp': '12',
      'cut off': 'NAN',
      'detector': 'weights/blur_jpg_prob0.1.pth',
      'model': 'stylegan2',
      'acc': 51.157500,
      'ap':53.487500
  }
  new_row_dict3 = {
      'bpp': '12',
      'cut off': 'NAN',
      'detector': 'weights/blur_jpg_prob0.5.pth',
      'model': 'stylegan2',
      'acc': 49.720000,
      'ap':52.080000
  }
  df_mitigation_freq_masking.loc[len(df_mitigation_freq_masking)] = new_row_dict1
  df_mitigation_freq_masking.loc[len(df_mitigation_freq_masking)] = new_row_dict2
  df_mitigation_freq_masking.loc[len(df_mitigation_freq_masking)] = new_row_dict3

  df_mitigation_freq_masking.to_csv(Path(RESULTS_SAVE_DIR) / "mitigation_results_freq_masking.csv", index=False)

df_mitigation_freq_masking

# Train

In [ ]:
def split_dataset(pilot_dataset:str, output_folder:str, bpp:str="bpp12", enable_val_set:bool=False):
    """
    Splits a nested image dataset into train/test or train/val/test folders.

    Args:
        pilot_dataset (str): Path to the root of the original dataset.
        output_folder (str): Path where the new dataset will be generated.
        bpp (str): The target bpp subfolder to use (e.g., 'bpp15', 'bpp50').
        enable_val_set (bool): If True, splits into train (80%), val (10%), test (10%).
                               If False, splits into train (80%), test (20%). Default is False.
    """


    bpp_path = os.path.join(pilot_dataset, bpp) # Path to the specific bpp directory

    if not os.path.exists(bpp_path): # Path existence check
        raise FileNotFoundError(f"Error: The specified path '{bpp_path}' does not exist.")

    def process_folder(folder_path, relative_path=""):
        """
        Recursively process folders to find class folders (0_real, 1_fake, etc.)
        with images inside. Handles arbitrary nesting levels.
        """
        try:
            entries = os.listdir(folder_path)
        except PermissionError:
            print(f"Warning: Permission denied for {folder_path}")
            return

        images = [f for f in entries if os.path.isfile(os.path.join(folder_path, f))]

        if images: # Check if this folder contains images directly
            if enable_val_set: # If validation set is enabled, the dataset is split into train (80%), val (10%), test (10%)
                train_imgs, temp_imgs = train_test_split(images, test_size=0.20, random_state=42)
                val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.50, random_state=42)

                def copy_files(image_list, split_type):
                    """Helper function to create directories and copy images"""
                    dest_dir = os.path.join(output_folder, split_type, relative_path)
                    os.makedirs(dest_dir, exist_ok=True)

                    for img in image_list:
                        src_file = os.path.join(folder_path, img)
                        dest_file = os.path.join(dest_dir, img)
                        shutil.copy2(src_file, dest_file)

                copy_files(train_imgs, 'train')
                copy_files(val_imgs, 'val')
                copy_files(test_imgs, 'test')
            else: # If validation set is disabled, the dataset is split into train (80%) and test (20%)
                train_imgs, test_imgs = train_test_split(images, test_size=0.20, random_state=42)

                def copy_files(image_list, split_type):
                    """Helper function to create directories and copy images"""
                    dest_dir = os.path.join(output_folder, split_type, relative_path)
                    os.makedirs(dest_dir, exist_ok=True)

                    for img in image_list:
                        src_file = os.path.join(folder_path, img)
                        dest_file = os.path.join(dest_dir, img)
                        shutil.copy2(src_file, dest_file)

                copy_files(train_imgs, 'train')
                copy_files(test_imgs, 'test')

            print(f"Processed: {relative_path} ({len(images)} images)")
        else: # If the current folder has no images inside, its subdirectories are checked
            subdirs = [d for d in entries if os.path.isdir(os.path.join(folder_path, d))]

            if subdirs:
                for subdir in subdirs:
                    subdir_path = os.path.join(folder_path, subdir)
                    new_relative_path = os.path.join(relative_path, subdir) if relative_path else subdir
                    process_folder(subdir_path, new_relative_path)
            else:
                if relative_path:
                    print(f"Warning: No images found in {folder_path}. Skipping.")

    categories = [d for d in os.listdir(bpp_path) if os.path.isdir(os.path.join(bpp_path, d))]
    split_info = "train/val/test (80/10/10)" if enable_val_set else "train/test (80/20)"

    print(f"Found {len(categories)} categories")
    print(f"Split mode: {split_info}")

    for category in categories:
        category_path = os.path.join(bpp_path, category)
        process_folder(category_path, category)

    print(f"\nDataset split complete! Files saved to: {output_folder}")

In [ ]:
def flatten_dataset(dataset):
    """Flattens a nested dataset structure into a single level with 0_real and 1_fake folders."""

    root = Path(dataset)

    original_dirs = [d for d in root.iterdir() if d.is_dir()]

    # Find every 0_real / 1_fake folder anywhere under root
    found = {
        "0_real": list(root.rglob("0_real")),
        "1_fake": list(root.rglob("1_fake")),
    }

    # Create the merged target folders.
    for label in ["0_real", "1_fake"]:
        (root / label).mkdir(exist_ok=True)

    # Move every image into the merged folders
    counts = {"0_real": 0, "1_fake": 0}
    for label, folders in found.items():
        for folder in folders:
            rel = folder.relative_to(root).parent
            prefix = "_".join(rel.parts) if rel.parts else "root"
            for img in folder.iterdir():
                if not img.is_file():
                    continue
                dst = root / label / f"{prefix}_{img.name}"
                shutil.move(str(img), str(dst))
                counts[label] += 1

    # Delete the now-empty original subfolders.
    for d in original_dirs:
        if d.name not in ("0_real", "1_fake"):
            shutil.rmtree(d)

    print(f"{root}: {counts}")

In [ ]:
def perform_CNNDetector_finetuning(dataset_folder:str, name:str = "unknown_finetuned_cnndetector"):
    """Performs finetuning of the CNNDetector on a given dataset folder."""

    if os.path.isdir(os.path.join(FINETUNED_MODELS_PATH, name)) and os.path.isfile(os.path.join(FINETUNED_MODELS_PATH, name, "model_epoch_best.pth")):
        print(f"Finetuned model {name} already available. Skipping...")
        return

    if not os.path.isdir(dataset_folder):
        print("⚠️ Warning! Requested dataset not available. Skipping finetuning...")
        return

    cmd = [
        "python",
        os.path.join(TOOLKIT_PATH, "train.py"),
        "--name", name,
        "--dataroot", dataset_folder,
        "--blur_prob", "0.5",
        "--blur_sig", "0.0,3.0",
        "--jpg_prob", "0.5",
        "--jpg_method", "cv2,pil",
        "--jpg_qual", "30,100"]

    print(f"--- Starting Finetuning: {name} ---")

    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    # Output
    for line in process.stdout:
        print(line, end='')

    process.wait()

    if process.returncode != 0:
        print(f"\n❌ CNNDetector Finetuning failed for {name} with return code {process.returncode}")
    else:
        print(f"\n✅ CNNDetector Finetuning process finished for {name}")
        print("Moving files to the finetuned_models directory...")
        !mkdir -p /content/input/finetuned_models
        !mv /content/checkpoints/{name} /content/input/finetuned_models/{name}
        print("Done! :)")

def perform_UnivFD_finetuning(dataset_folder:str, name:str = "unknown_finetuned_univfd"):
    """Performs finetuning of the UnivFD model on a given dataset folder."""
    
    if os.path.isdir(os.path.join(FINETUNED_MODELS_PATH, name)) and os.path.isfile(os.path.join(FINETUNED_MODELS_PATH, name, "model_epoch_best.pth")):
        print(f"Finetuned model {name} already available. Skipping...")
        return

    if not os.path.isdir(dataset_folder):
        print("⚠️ Warning! Requested dataset not available. Skipping finetuning...")
        return

    cmd = [
        "python",
        os.path.join(UNIVFD_SUITE_PATH, "train.py"),
        "--name", name,
        "--wang2020_data_path", dataset_folder,
        "--data_mode", "wang2020",
        "--arch", "CLIP:ViT-L/14",
        "--fix_backbone",
        "--batch_size=64",
        "--lr=0.0001",
        "--niter=10000",
        "--earlystop_epoch", "5"
    ]

    print(f"--- Starting UnivFD Finetuning: {name} ---")

    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    # Output
    for line in process.stdout:
        print(line, end='')

    process.wait()

    if process.returncode != 0:
        print(f"\n❌ UnivFD Finetuning failed for {name} with return code {process.returncode}")
    else:
        print(f"\n✅ UnivFD Finetuning process finished for {name}")
        print("Moving files to the finetuned_models directory...")
        !mkdir -p /content/input/finetuned_models
        !mv /content/checkpoints/{name} /content/input/finetuned_models/{name}
        print("Done! :)")

### Fine-Tuning Dataset

In [ ]:
if not os.path.isdir(JPEG_AI_DATASETS_PATH):
    print("Datasets folder not found. Generating splits from scratch.")

    os.makedirs(os.path.join("input", "JPEG_AI_datasets"), exist_ok=True)
    os.makedirs(os.path.join("input", "JPEG_AI_datasets", "unflattened"), exist_ok=True)

    print("Splitting bpp12...")
    split_dataset(DECOMPRESSED_IMG_BASE_PATH, os.path.join("input", "JPEG_AI_datasets", "unflattened", "bpp12_dataset"), "bpp12", True)
    print("Splitting bpp50...")
    split_dataset(DECOMPRESSED_IMG_BASE_PATH, os.path.join("input", "JPEG_AI_datasets", "unflattened", "bpp50_dataset"), "bpp50", True)
    print("Splitting bpp100...")
    split_dataset(DECOMPRESSED_IMG_BASE_PATH, os.path.join("input", "JPEG_AI_datasets", "unflattened", "bpp100_dataset"), "bpp100", True)

    # Datasets are flattened, in order to have a single 0_real and 1_fake folder per dataset
    print("Now proceeding with datasets flattening...")
    os.makedirs(os.path.join("input", "JPEG_AI_datasets", "flattened"), exist_ok=True)
    !cp -a ./input/JPEG_AI_datasets/unflattened/. ./input/JPEG_AI_datasets/flattened

    for dataset in ["bpp12_dataset", "bpp50_dataset", "bpp100_dataset"]:
        for split in ["train", "val", "test"]:
            print(f"Flattening {dataset}/{split}...")
            flatten_dataset(f"input/JPEG_AI_datasets/flattened/{dataset}/{split}")
    print("All done!")

### CNNDetector Fine-Tuning

In [ ]:
print("Finetuning CNNDetector using bpp12_dataset...")
perform_CNNDetector_finetuning(os.path.join(JPEG_AI_DATASETS_PATH, "flattened", "bpp12_dataset"), "bpp12_finetuned_cnndetector")
print("Finetuning CNNDetector using bpp50_dataset...")
perform_CNNDetector_finetuning(os.path.join(JPEG_AI_DATASETS_PATH, "flattened", "bpp50_dataset"), "bpp50_finetuned_cnndetector")
print("Finetuning CNNDetector using bpp100_dataset...")
perform_CNNDetector_finetuning(os.path.join(JPEG_AI_DATASETS_PATH, "flattened", "bpp100_dataset"), "bpp100_finetuned_cnndetector")
print("Finetuning CNNDetector using merged_dataset (bpp12+bpp50+bpp100)...")
perform_CNNDetector_finetuning(os.path.join(JPEG_AI_DATASETS_PATH, "flattened", "merged_dataset"), "merged_finetuned_cnndetector")

### UnivFD Fine-Tuning

In [ ]:
print("Finetuning UnivFD using bpp12_dataset...")
perform_UnivFD_finetuning(os.path.join(JPEG_AI_DATASETS_PATH, "unflattened", "bpp12_dataset"), "bpp12_finetuned_univfd")
print("Finetuning UnivFD using bpp50_dataset...")
perform_UnivFD_finetuning(os.path.join(JPEG_AI_DATASETS_PATH, "unflattened", "bpp50_dataset"), "bpp50_finetuned_univfd")
print("Finetuning UnivFD using bpp100_dataset...")
perform_UnivFD_finetuning(os.path.join(JPEG_AI_DATASETS_PATH, "unflattened", "bpp100_dataset"), "bpp100_finetuned_univfd")

# Test

In [ ]:
multi_level_test:bool = True # If set to True, each model will be tested on all possible BPPs. If set to False, each model will be tested only on its corresponding BPP dataset.

### CNNDetector Evaluation

In [ ]:
def evaluate_finetuned_cnndetector(model_pth:str, test_dataset_folder:str):
    """Performs evaluation of a finetuned CNNDetector model on a specified test dataset folder."""

    # Path existance checks
    if not os.path.exists(model_pth):
        raise FileNotFoundError(f"Model path '{model_pth}' does not exist.")
    if not os.path.exists(test_dataset_folder):
        raise FileNotFoundError(f"Test dataset folder '{test_dataset_folder}' does not exist.")

    test_detector_cnn(model_pth, test_dataset_folder, verbose=1)

    results_df = get_flat_rows(cnn_compression_results)
    if results_df.empty:
        print("⚠️ No results collected! Check dataset paths and structure.")
        return
    print(results_df[["detector", "model", "acc", "ap", "acc_real", "acc_fake"]])

In [ ]:
if not (REUSE_FT_CNN_DETECTOR and os.path.isfile(Path(RESULTS_SAVE_DIR) / "mitigation_results_finetuning_cnndetector.csv")):
    print("Evaluating finetuned CNNDetector models...")

    cnndetector_finetuned_models = ["bpp12_finetuned_cnndetector", "bpp50_finetuned_cnndetector", "bpp100_finetuned_cnndetector", "merged_finetuned_cnndetector"] # Models to evaluate

    # Initialization of results dictionary
    cnn_compression_results = {
        os.path.join(FINETUNED_MODELS_PATH, m, "model_epoch_best.pth"): {}
        for m in cnndetector_finetuned_models
    }
    CURRENT_IDS = []

    for model_name in cnndetector_finetuned_models:
        current_bpp = model_name.split("_")[0]  # Extract the BPP from the model name
        model_pth = os.path.join(FINETUNED_MODELS_PATH, model_name, "model_epoch_best.pth")

        if not os.path.isfile(model_pth):
            print(f"⚠️ Warning! The finetuned model {model_name} is not available. Skipping...")
            continue

        if not multi_level_test:
            print("Performing BPP-specific evaluation...")
            test_dataset_folder = os.path.join(JPEG_AI_DATASETS_PATH, "flattened", f"{current_bpp}_dataset", "test")
            print(f"\nEvaluating finetuned CNNDetector {model_name} on {current_bpp}_dataset...")
            evaluate_finetuned_cnndetector(model_pth, test_dataset_folder)
        else:
            print("Performing multi-level evaluation across all BPPs...")
            BPPs = ["bpp12", "bpp50", "bpp100", "merged"]
            for bpp in BPPs:
                test_dataset_folder = os.path.join(JPEG_AI_DATASETS_PATH, "flattened", f"{bpp}_dataset", "test")
                print(f"\nEvaluating finetuned CNNDetector {model_name} on {bpp}_dataset...")
                evaluate_finetuned_cnndetector(model_pth, test_dataset_folder)

    print("Done! :D")
else:
    print("Relying on pre-computed results.")

In [ ]:
if REUSE_FT_CNN_DETECTOR and os.path.isfile(Path(RESULTS_SAVE_DIR) / "mitigation_results_finetuning_cnndetector.csv"):
    df_cnndetector_finetuning_results = pd.read_csv(Path(RESULTS_SAVE_DIR) / "mitigation_results_finetuning_cnndetector.csv")
else:
    df_cnndetector_finetuning_results = get_flat_rows(cnn_compression_results) # Evaluation results for finetuned CNNDetector models
    df_cnndetector_finetuning_results.to_csv(Path(RESULTS_SAVE_DIR) / "mitigation_results_finetuning_cnndetector.csv", index=False)

df_cnndetector_finetuning_results

### UnivFD Evaluation

In [ ]:
def load_finetuned_univfd_model(model_name: str):
    """Loads finetuned UnivFD weights into the global model, freeing the previous one first."""

    global model, device

    # Explicitly free whatever is currently loaded (for memory efficiency)
    if 'model' in globals() and model is not None:
        del model
        torch.cuda.empty_cache()
        gc.collect()

    # Model loading
    model_pth = os.path.join(FINETUNED_MODELS_PATH, model_name, "model_epoch_best.pth")
    if not os.path.exists(model_pth):
        raise FileNotFoundError(f"Model not found: {model_pth}")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = get_model("CLIP:ViT-L/14")
    state_dict = torch.load(model_pth, map_location="cpu")
    model.load_state_dict(state_dict["model"])
    del state_dict   # free the raw checkpoint immediately
    gc.collect()
    model.eval()
    model.to(device)

    print(f"Loaded finetuned model: {model_name}")

In [ ]:
def evaluate_finetuned_univfd(model_name:str, test_dataset_folder:str):
    """Performs evaluation of a finetuned UnivFD model on a specified test dataset folder."""

    # Path existance checks
    model_path = os.path.join(FINETUNED_MODELS_PATH, model_name)
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model path '{model_path}' does not exist.")
    if not os.path.exists(test_dataset_folder):
        raise FileNotFoundError(f"Test dataset folder '{test_dataset_folder}' does not exist.")

    test_detector_univfd(model_name, test_dataset_folder)

    results_df = get_flat_rows(univfd_compression_results)
    if results_df.empty:
        print("⚠️ No results collected! Check dataset paths and structure.")
        return
    print(results_df[["detector", "model", "acc", "ap", "acc_real", "acc_fake"]])

In [ ]:
if not (REUSE_FT_UNIVFD and os.path.isfile(Path(RESULTS_SAVE_DIR) / "mitigation_results_finetuning_univfd.csv")):
    print("Evaluating finetuned UnivFD models...")

    univfd_finetuned_models = ["bpp12_finetuned_univfd", "bpp50_finetuned_univfd", "bpp100_finetuned_univfd"] # Models to evaluate

    # Initialization of results dictionary
    univfd_compression_results = {
        os.path.join(FINETUNED_MODELS_PATH, m, "model_epoch_best.pth"): {}
        for m in univfd_finetuned_models
    }
    CURRENT_IDS = []

    for model_name in univfd_finetuned_models:
        current_bpp = model_name.split("_")[0]  # Extract the BPP from the model name
        model_pth = os.path.join(FINETUNED_MODELS_PATH, model_name, "model_epoch_best.pth")

        if not os.path.isfile(model_pth):
            print(f"⚠️ Warning! The finetuned model {model_name} is not available. Skipping...")
            continue

        load_finetuned_univfd_model(model_name)

        if not multi_level_test:
            print("Performing BPP-specific evaluation...")
            test_dataset_folder = os.path.join(JPEG_AI_DATASETS_PATH, "unflattened", f"{current_bpp}_dataset", "test")
            print(f"\nEvaluating finetuned UnivFD {model_name} on {current_bpp}_dataset...")
            evaluate_finetuned_univfd(model_pth, test_dataset_folder)
        else:
            print("Performing multi-level evaluation across all BPPs...")
            BPPs = ["bpp12", "bpp50", "bpp100"]
            for bpp in BPPs:
                test_dataset_folder = os.path.join(JPEG_AI_DATASETS_PATH, "unflattened", f"{bpp}_dataset", "test")
                print(f"\nEvaluating finetuned UnivFD {model_name} on {bpp}_dataset...")
                evaluate_finetuned_univfd(model_pth, test_dataset_folder)

    print("Done! :D")
else:
    print("Relying on pre-computed results.")

In [ ]:
if REUSE_FT_UNIVFD and os.path.isfile(Path(RESULTS_SAVE_DIR) / "mitigation_results_finetuning_univfd.csv"):
    df_univfd_finetuning_results = pd.read_csv(Path(RESULTS_SAVE_DIR) / "mitigation_results_finetuning_univfd.csv")
else:
    df_univfd_finetuning_results:pd.DataFrame = get_flat_rows(univfd_compression_results) # Evaluation results for finetuned UnivFD models
    df_univfd_finetuning_results.to_csv(Path(RESULTS_SAVE_DIR) / "mitigation_results_finetuning_univfd.csv", index=False)

df_univfd_finetuning_results